In [1]:
%%configure
{
    "defaultLakehouse": {"name": "DE_LH_100_BondedWarehouse"}
}

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, -1, Finished, Available, Finished)

# **Establish Dataverse Shortcuts**
#### Description: This notebook will create required shortcuts utilising Semantic Links
#### 


##### Load required library

In [2]:
%pip install semantic-link-labs

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 7, Finished, Available, Finished)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 24.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.8 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 17.4 MB/s eta 0:00:00
  Attempting uninstall: semantic-link-sempy
    Found existing installation: semantic-link-sempy 0.8.0
    Not uninstalling semantic-link-sempy at /home/trusted-service-user/cluster-env/trident_env/lib/python3.10/site-packages, outside environment /nfs4/pyenv-0080fb1a-26b7-4fb2-9aea-cf1ca0b7731c
    Can't uninstall 'semantic-link-sempy'. No files were found to uninstall.

[notice] A new release of pip is available: 23.1.2 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [3]:
from pyspark.sql import Row
import sempy.fabric as fabric
import sempy_labs as sl

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 9, Finished, Available, Finished)

In [4]:
object_type = 'Lakehouse'
#object_name = 'DE_LH_100_BondedWarehouse'

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 10, Finished, Available, Finished)

#### Get lakehouse name

In [5]:
workspace_id = fabric.get_workspace_id()
workspace_name = fabric.resolve_workspace_name()

object_type = 'Lakehouse'
object_list = fabric.list_items(type = object_type, workspace = workspace_id)
object_name = object_list['Display Name'].values[0]

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 11, Finished, Available, Finished)

In [6]:
object_name

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 12, Finished, Available, Finished)

'DE_LH_100_BondedWarehouse'

In [7]:
# Check if 'DEV' or 'UAT' is in the workspace_name
if "DEV" in workspace_name or "UAT" in workspace_name:
    source_lakehouse = 'dataverse_endclothingt_cds2_workspace_unq0dcde15b8dcaee1190730022489f4'
else:
    source_lakehouse = 'dataverse_endclothing_cds2_workspace_unq6f6fff4b8c17ef119f85000d3a486'

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 13, Finished, Available, Finished)

##### Create shortcut for the metadata table organizationdatasyncfnostate

In [8]:
table_name = 'organizationdatasyncfnostate'

# Check if 'DEV' or 'UAT' is in the workspace_name
if "DEV" in workspace_name or "UAT" in workspace_name:
    source_workspace = '(UAT) Dataverse Sync'
    source_lakehouse = 'dataverse_endclothingt_cds2_workspace_unq0dcde15b8dcaee1190730022489f4'
else:
    source_workspace = 'Dataverse Sync'
    source_lakehouse = 'dataverse_endclothing_cds2_workspace_unq6f6fff4b8c17ef119f85000d3a486'

destination_lakehouse = object_name
destination_workspace = workspace_name


StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 14, Finished, Available, Finished)

In [9]:
#shortcut_name = 'organizationdatasyncfnostate'

#sl.lakehouse.create_shortcut_onelake(table_name, source_lakehouse, source_workspace, destination_lakehouse, destination_workspace, shortcut_name)

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 15, Finished, Available, Finished)

sempy_labs.lakehouse.delete_shortcut(shortcut_name: str, lakehouse: str | None = None, workspace: str | None = None)
Deletes a shortcut.

Parameters
:
shortcut_name (str) – The name of the shortcut.

lakehouse (str, default=None) – The Fabric lakehouse name in which the shortcut resides. Defaults to None which resolves to the lakehouse attached to the notebook.

workspace (str, default=None) – The name of the Fabric workspace in which lakehouse resides. Defaults to None which resolves to the workspace of the attached lakehouse or if no lakehouse attached, resolves to the workspace of the notebook.

### Step 04
##### Create table shortcuts, if they exist they will be dropped first
Description:

The overall goal of this script is to manage shortcuts in a lakehouse system by first attempting to delete any existing shortcuts and then creating new ones, excluding any entries named 'metadata'. The process involves reading a list of entities from a Spark table and using their names as identifiers for the shortcuts.








In [10]:
#df = spark.read.table("organizationdatasyncfnostate").select('entitynamename')

# List of table names in lowercase
table_names = [
    "custpackingslipjour",
    "custpackingsliptrans",
    "custtable",
    "dirpartytable",
    "ecorescategory",
    "ecorescategoryintrastat",
    "ecoresproduct",
    "ecoresproducttranslation",
    "hslcommercialinvoice",
    "inventiteminventsetup",
    "inventjournaltable",
    "inventjournaltrans",
    "inventsum",
    "inventtable",
    "inventtablemodule",
    "logisticsaddresscountryregion",
    "logisticspostaladdress",
    "purchtable",
    "salesline",
    "salestable",
    "vendpackingslipjour",
    "vendpackingsliptrans",
    "vendtable",
    "whscontainerline",
    "whscontainertable",
    "whsloadline",
    "whsloadtable",
    "whsshipmenttable"
]

#    "ecorescategorycommoditycode",

# Create a list of Rows
rows = [Row(entitynamename=name) for name in table_names]

# Create a DataFrame from the Rows
df = spark.createDataFrame(rows)


StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 16, Finished, Available, Finished)

In [11]:
for row in df.collect():
    if row['entitynamename'] is not None:
        print(row['entitynamename'])
        shortcut_name = row['entitynamename']
        try:
            sl.lakehouse.delete_shortcut(shortcut_name, destination_lakehouse, destination_workspace)
        except Exception as e:
            print(f"{shortcut_name} - not found: {str(e)}")

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 17, Finished, Available, Finished)

custpackingslipjour
custpackingslipjour - not found: 404 Not Found for url: https://api.fabric.microsoft.com//v1/workspaces/ad0573da-93d1-489b-8498-58a042058210/items/b00d0325-3b91-4ecd-a3fb-f846284d0329/shortcuts/Tables/custpackingslipjour
Error: {"requestId":"2f90a947-7844-436f-b54f-f8630e7eddf2","errorCode":"EntityNotFound","moreDetails":[{"errorCode":"ShortcutNotFound","message":"Shortcut custpackingslipjour is not found"}],"message":"The requested resource could not be found"}
Headers: {'Cache-Control': 'no-store, must-revalidate, no-cache', 'Pragma': 'no-cache', 'Transfer-Encoding': 'chunked', 'Content-Type': 'application/json; charset=utf-8', 'x-ms-public-api-error-code': 'EntityNotFound', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'X-Frame-Options': 'deny', 'X-Content-Type-Options': 'nosniff', 'RequestId': '2f90a947-7844-436f-b54f-f8630e7eddf2', 'Access-Control-Expose-Headers': 'RequestId', 'request-redirected': 'true', 'home-cluster-uri': 'https://wabi

In [12]:
for row in df.collect():
    if row['entitynamename'] is not None:
        print(row['entitynamename'])
        table_name = row['entitynamename']
        shortcut_name = table_name
        if table_name != 'metadata':
            try:
                sl.lakehouse.create_shortcut_onelake(table_name, source_lakehouse, source_workspace, destination_lakehouse, destination_workspace, shortcut_name)
            except Exception as e:
                print(f"Error creating shortcut for {table_name}: {str(e)}")

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 18, Finished, Available, Finished)

custpackingslipjour
🟢 The shortcut 'custpackingslipjour' was created in the 'DE_LH_100_BondedWarehouse' lakehouse within the '(DEV) Bonded Warehouse workspace. It is based on the 'custpackingslipjour' table in the 'dataverse_endclothingt_cds2_workspace_unq0dcde15b8dcaee1190730022489f4' lakehouse within the '(UAT) Dataverse Sync' workspace.
custpackingsliptrans
🟢 The shortcut 'custpackingsliptrans' was created in the 'DE_LH_100_BondedWarehouse' lakehouse within the '(DEV) Bonded Warehouse workspace. It is based on the 'custpackingsliptrans' table in the 'dataverse_endclothingt_cds2_workspace_unq0dcde15b8dcaee1190730022489f4' lakehouse within the '(UAT) Dataverse Sync' workspace.
custtable
🟢 The shortcut 'custtable' was created in the 'DE_LH_100_BondedWarehouse' lakehouse within the '(DEV) Bonded Warehouse workspace. It is based on the 'custtable' table in the 'dataverse_endclothingt_cds2_workspace_unq0dcde15b8dcaee1190730022489f4' lakehouse within the '(UAT) Dataverse Sync' workspace.
d

In [13]:
# for row in df.collect():
#     print(row['entitynamename'])
#     shortcut_name = row['entitynamename']
#     try:
#         sl.lakehouse.delete_shortcut(shortcut_name,destination_lakehouse,destination_workspace)
#     except:
#         print(f"{shortcut_name} - not found")

# for row in df.collect():
#     print(row['entitynamename'])
#     table_name = row['entitynamename']
#     shortcut_name = table_name
#     if table_name != 'metadata':
#         sl.lakehouse.create_shortcut_onelake(table_name, source_lakehouse, source_workspace, destination_lakehouse, destination_workspace, shortcut_name)

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 19, Finished, Available, Finished)

### Step 05
##### Test cells

In [14]:
# table_name = 'salesline'
# source_lakehouse = 'dataverse_endclothingt_cds2_workspace_unq0dcde15b8dcaee1190730022489f4'
# source_workspace = '(UAT) Dataverse Sync'
# shortcut_name = 'salesline'

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 20, Finished, Available, Finished)

In [15]:
# sl.lakehouse.create_shortcut_onelake(table_name, source_lakehouse, source_workspace, destination_lakehouse, destination_workspace, shortcut_name)

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 21, Finished, Available, Finished)

In [16]:
# https://onelake.dfs.fabric.microsoft.com/UAT_DataverseSync/dataverse_endclothingt_cds2_workspace_unq0dcde15b8dcaee1190730022489f4.Lakehouse/Tables/salesline

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 22, Finished, Available, Finished)

In [17]:
# sl.lakehouse.delete_shortcut(shortcut_name,destination_lakehouse,destination_workspace)

StatementMeta(, face1095-b247-4ebf-a778-61962b9f3cbe, 23, Finished, Available, Finished)